# Mixed layer depth in ACCESS models and observations

Related issue: https://github.com/ACCESS-Community-Hub/access-om3-paper-1/issues/38

Requires access to `/g/data/av17`

In [ ]:
# These first two cells must be in all notebooks!
# It allows us to run all the notebooks at once, this cell has a tag "parameters" which allows us to pass in 
# arguments externally using papermill (see mkfigs.sh for details)

# Set esm_file to the datastore for the main experiment of interest

# Second experiment to compare against (set to None to skip the comparison)
esm_file_compare = None
esm_file = "/g/data/ol01/outputs/access-om3-25km/MC_25km_jra_iaf+wombatlite-test3v2-00532b88/datastore.json"
# esm_file = "/g/data/zv30/non-cmip/ACCESS-CM3/cm3-run-27-07-2026-PD-control/cm3-datastore/cm3-datastore.json"

# What physical field you want, in CF terms (stays the same across models)
variable_standard_name = "Ocean Mixed Layer Thickness Defined by Sigma T"

# Fallback variable name to use if the catalog doesn’t expose standard_name
fallback_variable_names = ["mlotst", "mld", "mixed_layer_depth", "mixed_layer_thickness"]

# Frequency depends on the datastore you’re using:
data_frequency = "1mon"

# papermill settings. *No need to modify these if running interactively.* 
papermill = False                      # `cwd` and `nbname` will be populated by papermill.
cwd = None                             # current working directory 
nbname = None                          # notebook name

In [ ]:
# Parameters
esm_file = "/g/data/ik11/outputs/access-om3-025/MC_25km_jra_ryf+wombatlite-test5.1-5c6ad028/datastore.json"
papermill = True
cwd = "/g/data/v45/aek156/notebooks/github/ACCESS-Community-Hub/access-om3-paper-1/notebooks/mkfigs_output_MC_25km_jra_ryf+wombatlite-test5.1-5c6ad028/"
nbname = "MLD.ipynb"


In [ ]:
if not papermill: 
    import nci_ipynb, os  # requires conda/analysis3-26.03 or later
    cwd = nci_ipynb.dir()
    nbname = nci_ipynb.name()
    os.chdir(cwd)
import mkfigs_bootstrap  # noqa: adds external/access-model-mkfigs/src to sys.path (stop-gap)
from mkfigs import MkmdWriter
mkmd = MkmdWriter(esm_file, nbname, str(cwd), pm=papermill)
from exptdata_access import get_experiment_info, guess_experiment_from_esm_file
from model_agnostic import get_lon_lat_from_catalog, select_variable, patch_broken_conda_env
patch_broken_conda_env()  # conda env workaround for analysis3-26.03+; no-op on healthy envs
# Infer model information from the datastore path
expt_key, info = guess_experiment_from_esm_file(esm_file)
model_name = info["model"]             # OM3 or CM3

In [ ]:
import xarray as xr
import cf_xarray as cfxr
import cf_xarray.units
import pint_xarray
from pint import application_registry as ureg
import intake
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
from distributed import Client
import cftime
import os
import matplotlib.pyplot as plt
import cmocean as cm
import cartopy.crs as ccrs
import cartopy.feature as cft
from textwrap import wrap
xr.set_options(keep_attrs=True);  # cf_xarray works best when xarray keeps attributes by default

### Define plot function

In [ ]:
blue_marble = plt.imread('/g/data/ik11/grids/BlueMarble.tiff')
blue_marble_extent = (-180, 180, -90, 90)

In [ ]:
def plot(dat, title=None, **kwargs):
    if title is None:
        title = dat.attrs['long_name']
    fig = plt.figure(figsize=(12, 6))
    ax = plt.axes(projection=ccrs.Robinson(central_longitude=-100))
    if "geolon" in dat.coords:  # OM3 native grid: plot at true positions (xh/yh are nominal north of ~65N)
        kwargs.setdefault("x", "geolon")
        kwargs.setdefault("y", "geolat")
    dat.plot.contourf(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cbar_kwargs={"label": "\n".join(wrap(f"{dat.attrs['long_name']} [{dat.attrs['units']}]", 45)),
                     "fraction": 0.03, "aspect": 15, "shrink": 0.7},
        **kwargs
    )
    
    # Add blue marble land:
    ax.imshow(
        blue_marble, extent=blue_marble_extent, transform=ccrs.PlateCarree(), origin="upper"
    )
    
    plt.title(title);

In [ ]:
def seasonal_mean(da, months, weights):
    # Select months, then do time-weighted mean
    da_m = da.where(da["time"].dt.month.isin(months), drop=True)
    w_m  = weights.where(da["time"].dt.month.isin(months), drop=True)
    return da_m.weighted(w_m).mean("time")

### Load and plot data from ACCESS-OM3

In [ ]:
from exptdata_access import get_exptname_from_path
exptname=get_exptname_from_path(esm_file)
print("Experiment name:", exptname)

datastore = intake.open_esm_datastore(
    esm_file,
    columns_with_iterables=[
        "variable",
        "variable_long_name",
        "variable_standard_name",
        "variable_cell_methods",
        "variable_units"
    ]
)

In [ ]:
try:
    from distributed import get_client
    get_client().close()
except ValueError:
    pass
if "client" in globals():
    del client

da_model = select_variable(
    datastore,
    variable_standard_name,
    fallback_variable_names,
    data_frequency=data_frequency,
)
print("Selected variable:", da_model.name)
print("dims:", da_model.dims)

In [ ]:
# Discover lon/lat from the catalog (using the helper we wrote earlier)
lon, lat = get_lon_lat_from_catalog(datastore)

# Attach the true 2-D lon/lat as their own coords (used by plot()). Don't skip this when
# da_model.cf["longitude"] exists: that finds the 1-D nominal xh/yh, which are not
# geographic north of the tripolar join (~65N).
model_all = da_model.assign_coords(geolon=lon, geolat=lat)
print("Attached geolon/geolat from catalog grid variables.")

print("model_all dims:", model_all.dims)

In [ ]:
# Averaging window: last NYEARS complete years, or an explicit [first, last] period
averaging_period = None   # e.g. [1908, 1909]; None = automatic
NYEARS = 10
from model_agnostic import years_window, year_slice

In [ ]:
if exptname == "25km-iaf-test-for-AK-expt-7df5ef4c":
    model_all = model_all.isel(yh=slice(10, None))
    lon = lon.isel(yh=slice(10, None))
    lat = lat.isel(yh=slice(10, None))

In [ ]:
# Inspect full time coverage
print("Full model time range:", model_all.time.values[0], "→", model_all.time.values[-1])

# Choose the window on the native time axis, before any calendar conversion
# (converting daily noleap data to proleptic_gregorian would make leap years look incomplete)
om3_window = years_window(model_all.time, NYEARS, period=averaging_period)
print("Averaging window:", om3_window)
period_label = f"{om3_window[0]}-{om3_window[1]}"

# Restrict model to this window, then put it on a common calendar
model_window = model_all.sel(time=year_slice(om3_window))
model_window = model_window.convert_calendar("proleptic_gregorian", use_cftime=True)
print("Windowed dims:", model_window.dims)

In [ ]:
# weights = days in each timestep's month
w = model_window["time"].dt.days_in_month

In [ ]:
%%time
model_JFM_mean = seasonal_mean(model_window, [1, 2, 3], w).compute()
model_JAS_mean = seasonal_mean(model_window, [7, 8, 9], w).compute()

In [ ]:
plot(model_JFM_mean,
    levels=51,
    vmin=0,
    vmax=None,
    cmap='viridis',
    title=f"{model_window.attrs['long_name']} JFM {period_label} mean\nin {exptname}"
    )

In [ ]:
plot(model_JAS_mean,
    levels=51,
    vmin=0,
    vmax=None,
    cmap='viridis',
    title=f"{model_window.attrs['long_name']} JAS {period_label} mean\nin {exptname}"
    )

In [ ]:
plot(model_JFM_mean,
    levels=51,
    vmin=0,
    vmax=500,
    extend="max",
    cmap='viridis',
    title=f"{model_window.attrs['long_name']} JFM {period_label} mean\nin {exptname}"
    )
mkmd.savefig(plt.gcf(), "Mixed Layer Depth: Model JFM", "Ocean Mixed Layer Depth JFM. GitHub issue: [#38](https://github.com/ACCESS-Community-Hub/access-om3-paper-1/issues/38) — Time-max mixed layer depth maps")

In [ ]:
plot(model_JAS_mean,
    levels=51,
    vmin=0,
    vmax=500,
    extend="max",
    cmap='viridis',
    title=f"{model_window.attrs['long_name']} JAS {period_label} mean\nin {exptname}"
    )
mkmd.savefig(plt.gcf(), "Mixed Layer Depth: Model JAS", "Ocean Mixed Layer Depth JAS. GitHub issue: [#38](https://github.com/ACCESS-Community-Hub/access-om3-paper-1/issues/38) — Time-max mixed layer depth maps")

### Load and plot data from MC comparison experiment

In [ ]:
if esm_file_compare is not None:
    exptname_compare = get_exptname_from_path(esm_file_compare)
    print("Comparison experiment name:", exptname_compare)

    datastore_compare = intake.open_esm_datastore(
        esm_file_compare,
        columns_with_iterables=[
            "variable",
            "variable_long_name",
            "variable_standard_name",
            "variable_cell_methods",
            "variable_units"
        ]
    )

    da_model_compare = select_variable(
        datastore_compare,
        variable_standard_name,
        fallback_variable_names,
        data_frequency=data_frequency,
    )
    print("Selected variable:", da_model_compare.name)
    print("dims:", da_model_compare.dims)

In [ ]:
if esm_file_compare is not None:
    # Discover lon/lat from the comparison catalog
    lon_compare, lat_compare = get_lon_lat_from_catalog(datastore_compare)

    # true 2-D lon/lat for plot(); xh/yh are nominal north of ~65N
    model_all_compare = da_model_compare.assign_coords(geolon=lon_compare, geolat=lat_compare)
    print("Attached geolon/geolat from catalog grid variables.")

    print("model_all_compare dims:", model_all_compare.dims)

In [ ]:
if esm_file_compare is not None:
    print("Full comparison model time range:",
          model_all_compare.time.values[0], "→", model_all_compare.time.values[-1])

    # Use the same model years as the main experiment where the comparison run has them, so
    # differences between the two are not confounded by different periods. years_window warns
    # if only part is covered, and falls back to the comparison run's first NYEARS complete
    # years (with a warning) if none is. Chosen on the native time axis, before any conversion.
    compare_window = years_window(model_all_compare.time, NYEARS, within=om3_window, fallback="last")
    print("Comparison averaging window:", compare_window)
    period_label_compare = f"{compare_window[0]}-{compare_window[1]}"

    model_window_compare = model_all_compare.sel(time=year_slice(compare_window))
    model_window_compare = model_window_compare.convert_calendar("proleptic_gregorian", use_cftime=True)
    print("Windowed dims:", model_window_compare.dims)

    # weights = days in each timestep's month
    w_compare = model_window_compare["time"].dt.days_in_month

    model_compare_JFM_mean = seasonal_mean(model_window_compare, [1, 2, 3], w_compare).compute()
    model_compare_JAS_mean = seasonal_mean(model_window_compare, [7, 8, 9], w_compare).compute()

In [ ]:
if esm_file_compare is not None:
    plot(model_compare_JFM_mean,
        levels=51,
        vmin=0,
        vmax=500,
        extend="max",
        cmap='viridis',
        title=f"{model_window_compare.attrs['long_name']} JFM {period_label_compare} mean\nin {exptname_compare}"
        )
    mkmd.savefig(plt.gcf(), "Mixed Layer Depth: Comparison JFM", "Ocean Mixed Layer Depth JFM (comparison experiment). GitHub issue: [#38](https://github.com/ACCESS-Community-Hub/access-om3-paper-1/issues/38) — Time-max mixed layer depth maps")

In [ ]:
if esm_file_compare is not None:
    plot(model_compare_JAS_mean,
        levels=51,
        vmin=0,
        vmax=500,
        extend="max",
        cmap='viridis',
        title=f"{model_window_compare.attrs['long_name']} JAS {period_label_compare} mean\nin {exptname_compare}"
        )
    mkmd.savefig(plt.gcf(), "Mixed Layer Depth: Comparison JAS", "Ocean Mixed Layer Depth JAS (comparison experiment). GitHub issue: [#38](https://github.com/ACCESS-Community-Hub/access-om3-paper-1/issues/38) — Time-max mixed layer depth maps")

### Load and plot obs data from DeBoyer Montegut (2023)
https://doi.org/10.17882/91774

In [ ]:
MLDobs = xr.open_dataset('/g/data/av17/access-nri/OM3/MLD-DeBoyerMontegut2023/mld_dr003_ref10m_v2023.nc')['mld_dr003']
MLDobs.attrs['units'] = MLDobs.attrs['unit']  # fix so plot works

# TODO: append copy of westernmost data to eastern end to avoid gap in plot

In [ ]:
# small BUG: mean of monthly means is not mean of days in that month (eg Feb gets slightly more heavily weighted)
MLDobs_JFM_mean = MLDobs.isel(time=[0, 1, 2]).mean('time').load()
MLDobs_JAS_mean = MLDobs.isel(time=[6, 7, 8]).mean('time').load()

In [ ]:
plot(MLDobs_JFM_mean,
    levels=51,
    vmin=0,
    vmax=500,
    extend="max",
    cmap='viridis',
    title=f"Observed mixed layer depth JFM mean (DeBoyer Montegut, 2023)"
    )
mkmd.savefig(plt.gcf(), "Mixed Layer Depth: Observed JFM", "Observed ocean Mixed Layer Depth JFM. GitHub issue: [#38](https://github.com/ACCESS-Community-Hub/access-om3-paper-1/issues/38) — Time-max mixed layer depth maps")

In [ ]:
plot(MLDobs_JAS_mean,
    levels=51,
    vmin=0,
    vmax=500,
    extend="max",
    cmap='viridis',
    title=f"Observed mixed layer depth JAS mean (DeBoyer Montegut, 2023)"
    )
mkmd.savefig(plt.gcf(), "Mixed Layer Depth: Observed JAS", "Observed ocean Mixed Layer Depth JAS. GitHub issue: [#38](https://github.com/ACCESS-Community-Hub/access-om3-paper-1/issues/38) — Time-max mixed layer depth maps")